ĐỀ BÀI: [https://apicrm.cybersoft.edu.vn/files/03-03-2026-02-12-18-[capstone]-chuong-2_-fine-tuning-llama-voi-lora_qlora-cho-phan-tich-tai-chinh.pdf](https://)

1. CÀI ĐẶT THƯ VIỆN

In [1]:
!pip install -qq --upgrade pip
!pip install -qq --upgrade peft transformers accelerate bitsandbytes datasets trl huggingface_hub
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


2. LOAD PACKAGES

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


import torch
import numpy as np

# Nhập các lớp và hàm cụ thể từ thư viện PEFT (Parameter-Efficient Fine-Tuning) để tinh chỉnh hiệu quả.
from peft import PeftModel, PeftConfig, LoraConfig, TaskType, get_peft_model, get_peft_config
# Nhập AutoModelForCausalLM và AutoTokenizer từ thư viện Hugging Face transformers để tải các mô hình và bộ mã hóa đã được đào tạo trước.
from transformers import AutoModelForCausalLM, AutoTokenizer
# Nhập DataCollatorForLanguageModeling, Trainer và TrainingArguments từ transformers để chuẩn bị và huấn luyện dữ liệu.
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
# Nhập các class Dataset, DatasetDict, Features, Value, ClassLabel để tự tạo dataset từ file txt Financial PhraseBank.
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
# Nhập SFTTrainer từ thư viện TRL (Transformer Reinforcement Learning) để tinh chỉnh có giám sát.
from trl import SFTTrainer
# Nhập BitsAndBytesConfig từ transformers để cấu hình lượng tử hóa 4-bit hoặc 8-bit.
from transformers import BitsAndBytesConfig
# Nhập prepare_model_for_kbit_training từ PEFT để chuẩn bị mô hình đã lượng tử hóa cho quá trình đào tạo k-bit.
from peft import prepare_model_for_kbit_training
# Nhập thư viện evaluate, cung cấp giao diện đơn giản để đánh giá các chỉ số.
import evaluate
# Nhập mô-đun warnings để kiểm soát cách xử lý các cảnh báo.
import warnings

# Cấu hình mô-đun warnings để bỏ qua tất cả các cảnh báo trong quá trình thực thi.
warnings.filterwarnings("ignore")


3. THIẾT LẬP CẤU HÌNH

In [4]:
# Định nghĩa định danh của mô hình Llama 3.2 1B Instruct đã được đào tạo trước từ Hugging Face.
base_model_id = "unsloth/Llama-3.2-1B-Instruct"
# Định nghĩa thư mục cục bộ nơi các mô hình và bộ dữ liệu đã tải xuống sẽ được lưu trữ.
cache_dir = "./cache"
output_dir = "/content/drive/MyDrive/CYBERSOFT/BAI_TAP/QLORA/CHECKPOINT"
output_logs = "/content/drive/MyDrive/CYBERSOFT/BAI_TAP/QLORA/LOGS"

# Tải mô hình cơ sở QLoRA với lượng tử hóa 4-bit
# Khởi tạo đối tượng BitsAndBytesConfig để thiết lập lượng tử hóa 4-bit cho mô hình cơ sở.
quantization_config = BitsAndBytesConfig(
    # Chỉ định rằng trọng số của mô hình nên được tải ở độ chính xác 4-bit.
    load_in_4bit=True,
    # Đặt loại lượng tử hóa 4-bit thành "nf4" (NormalFloat4).
    bnb_4bit_quant_type="nf4",
    # Bật lượng tử hóa kép, lượng tử hóa chính các hằng số lượng tử hóa.
    bnb_4bit_use_double_quant=True,
    # bnb_4bit_compute_dtype=torch.bfloat16, # Google colab does not support bfloat16
    # Đặt kiểu dữ liệu cho các phép tính trong quá trình lượng tử hóa thành torch.float16 (số dấu phẩy động nửa chính xác).
    bnb_4bit_compute_dtype=torch.float16,
)

In [5]:
MAX_TRAIN_STEPS = 800 # Định nghĩa số bước huấn luyện tối đa. Dataset Financial PhraseBank nhỏ nên giảm để tránh overfitting.
NUM_EVAL_STEPS = 100 # Định nghĩa số bước đánh giá giữa các lần lưu mô hình hoặc ghi nhật ký.
MAX_TRAIN_SAMPLES = 10_000 # Định nghĩa số lượng mẫu huấn luyện tối đa được sử dụng.
MAX_EVAL_SAMPLES = 1_000 # Định nghĩa số lượng mẫu đánh giá tối đa được sử dụng.

training_args = TrainingArguments(
    output_dir=output_dir, # Thư mục để lưu trữ các checkpoint của mô hình và đầu ra đào tạo.
    # num_train_epochs=1, # Số lượng epoch huấn luyện. Tạm thời bị bỏ qua vì 'max_steps' được sử dụng.
    per_device_train_batch_size=8, # Kích thước batch cho mỗi thiết bị trong quá trình huấn luyện.
    per_device_eval_batch_size=8, # Kích thước batch cho mỗi thiết bị trong quá trình đánh giá.
    logging_dir=output_logs, # Thư mục để lưu trữ nhật ký TensorBoard.
    logging_steps=10, # Số bước giữa các lần ghi nhật ký tiến độ huấn luyện.
    save_steps=NUM_EVAL_STEPS, # Số bước giữa các lần lưu checkpoint của mô hình.
    max_steps=MAX_TRAIN_STEPS, # Tổng số bước huấn luyện tối đa. Khi đạt đến, quá trình đào tạo sẽ dừng lại.
    eval_steps=NUM_EVAL_STEPS, # Số bước giữa các lần chạy đánh giá.
    eval_strategy="steps", # Chiến lược để chạy đánh giá (chạy theo số bước).
    save_strategy="steps", # Chiến lược lưu checkpoint theo số bước để khớp với eval_strategy.
    # overwrite_output_dir=True, # Ghi đè thư mục đầu ra nếu nó đã tồn tại.
    save_total_limit=2, # Giới hạn tổng số checkpoint mô hình được lưu. Checkpoint cũ hơn sẽ bị xóa.
    report_to="none", # Không báo cáo kết quả đào tạo cho bất kỳ nền tảng nào.
    push_to_hub=False, # Không đẩy mô hình lên Hugging Face Hub.
    remove_unused_columns=False, # Không xóa các cột không được sử dụng bởi mô hình từ bộ dữ liệu.
    load_best_model_at_end=True, # Tự động load checkpoint tốt nhất ở cuối quá trình train.
    metric_for_best_model="f1", # Dùng F1 để chọn checkpoint tốt nhất vì dataset bị lệch nhãn.
    greater_is_better=True, # F1 càng cao càng tốt.
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


4. LOAD BASE MODEL

In [6]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True, cache_dir=cache_dir) # Tải bộ mã hóa (tokenizer) từ mô hình cơ sở đã được xác định. trust_remote_code=True cho phép mã tùy chỉnh và cache_dir chỉ định thư mục lưu trữ cục bộ.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, # Định danh của mô hình cơ sở sẽ được tải.
    trust_remote_code=True, # Cho phép tải và thực thi mã Python tùy chỉnh từ kho lưu trữ của mô hình.
    cache_dir=cache_dir, # Thư mục để lưu trữ các tệp mô hình đã tải xuống.
    quantization_config=quantization_config, # Áp dụng cấu hình lượng tử hóa 4-bit đã được định nghĩa trước.
    device_map="cuda:0" if torch.cuda.is_available() else "cpu", # Đặt mô hình trên GPU nếu có sẵn, nếu không thì dùng CPU.
)
base_model = prepare_model_for_kbit_training(base_model) # Chuẩn bị mô hình đã được lượng tử hóa để đào tạo k-bit, đảm bảo các tham số PEFT được định cấu hình đúng cách.

base_model # Hiển thị cấu trúc của mô hình cơ sở sau khi tải và chuẩn bị.

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    

In [7]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True, cache_dir=cache_dir) # Tải bộ mã hóa (tokenizer) từ mô hình cơ sở đã được xác định. trust_remote_code=True cho phép mã tùy chỉnh và cache_dir chỉ định thư mục lưu trữ cục bộ.

In [8]:
if tokenizer.pad_token is None or tokenizer.pad_token_id is None:
    print("Pad token is not set. Setting it to EOS token.") # In ra thông báo rằng 'pad token' chưa được đặt và sẽ được đặt thành 'EOS token'.
    tokenizer.pad_token = tokenizer.eos_token # Đặt 'pad token' của bộ mã hóa bằng 'eos token'.
    tokenizer.pad_token_id = tokenizer.eos_token_id # Đặt 'pad token ID' của bộ mã hóa bằng 'eos token ID'.
else:
    print(f'Pad token: {tokenizer.pad_token}') # Nếu 'pad token' đã được đặt, in ra giá trị của nó.
    print(f'Pad token id: {tokenizer.pad_token_id}') # Nếu 'pad token' đã được đặt, in ra ID của nó.

print(f'EOS token: {tokenizer.eos_token}') # In ra giá trị của 'EOS token' (End Of Sentence token).
print(f'EOS token id: {tokenizer.eos_token_id}') # In ra ID của 'EOS token'.

Pad token: <|finetune_right_pad_id|>
Pad token id: 128004
EOS token: <|eot_id|>
EOS token id: 128009


5. LOAD AND APPLY LoRA

In [9]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # Chỉ định loại tác vụ cho mô hình (ví dụ: mô hình ngôn ngữ nhân quả).
    inference_mode=False, # Đặt chế độ đào tạo; False có nghĩa là mô hình đang trong chế độ huấn luyện (có thể cập nhật trọng số).
    r=8, # Đặt thứ hạng cho ma trận LoRA. Giá trị nhỏ hơn giúp giảm số lượng tham số có thể đào tạo.
    lora_alpha=32, # Đặt hệ số tỷ lệ cho ma trận LoRA. lora_alpha càng lớn thì trọng số càng tác động nhiều hơn.
    lora_dropout=0.1 # Đặt tỷ lệ dropout cho các lớp LoRA để tránh overfitting.
)

In [10]:
peft_model = get_peft_model(base_model, peft_config) # Tạo mô hình PEFT bằng cách truyền mô hình cơ sở và cấu hình PEFT.
peft_model.print_trainable_parameters() # In ra số lượng tham số có thể huấn luyện và tổng số tham số của mô hình, cùng với tỷ lệ phần trăm.
peft_model # Hiển thị cấu trúc của mô hình PEFT sau khi được tạo.

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
          

6. DOWNLOAD DATASETS

In [12]:
import zipfile
import requests
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

dataset_url = (
    "https://huggingface.co/datasets/"
    "takala/financial_phrasebank/resolve/main/"
    "data/FinancialPhraseBank-v1.0.zip"
)

# Dùng Sentences_50Agree.txt để có nhiều dữ liệu nhất, giảm overfitting so với Sentences_AllAgree.txt.
DATASET_FILE_NAME = "Sentences_50Agree.txt"


In [13]:
download_dir = os.path.join(cache_dir, "financial_phrasebank")

In [14]:
os.makedirs(download_dir, exist_ok=True)

In [15]:
zip_path = os.path.join(
    download_dir,
    "FinancialPhraseBank-v1.0.zip"
)

In [16]:
response = requests.get(dataset_url)
response.raise_for_status() # Kiểm tra lỗi tải file.

In [17]:
with open(zip_path, "wb") as f:
    f.write(response.content)

In [18]:
extract_dir = os.path.join(
    download_dir,
    "FinancialPhraseBank-v1.0"
)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(download_dir)

In [19]:
file_path = os.path.join(
    extract_dir,
    DATASET_FILE_NAME
)

print("Using dataset file:", file_path)

Using dataset file: ./cache/financial_phrasebank/FinancialPhraseBank-v1.0/Sentences_50Agree.txt


In [20]:
data = []

with open(file_path, "r", encoding="latin-1") as f:

    for line in f:

        if not line.strip():
            continue

        sentence, label = line.rsplit("@", 1)

        data.append({
            "sentence": sentence.strip(),
            "label": label.strip()
        })

print("Total samples:", len(data))

Total samples: 4846


In [21]:
data[0]

{'sentence': 'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .',
 'label': 'neutral'}

In [22]:
all_labels = ["negative", "neutral", "positive"] # Cố định thứ tự nhãn để tránh mapping bị thay đổi giữa các lần chạy.

In [23]:
label2id = {label: i for i, label in enumerate(all_labels)} # Tạo một ánh xạ từ tên nhãn sang ID số nguyên.
id2label = {i: label for i, label in enumerate(all_labels)} # Tạo một ánh xạ ngược từ ID số nguyên sang tên nhãn.

In [24]:
label2id

{'negative': 0, 'neutral': 1, 'positive': 2}

In [25]:
id2label

{0: 'negative', 1: 'neutral', 2: 'positive'}

In [26]:
processed_data = [
    {
        "sentence": item["sentence"],
        "label": item["label"]
    }
    for item in data
]

In [27]:
features = Features({
    "sentence": Value("string"),
    "label": ClassLabel(names=all_labels),
})

dataset = Dataset.from_list(processed_data, features=features)

dataset

Dataset({
    features: ['sentence', 'label'],
    num_rows: 4846
})

In [28]:
# Giới hạn số lượng mẫu nếu cần. Với Sentences_50Agree.txt, dataset chỉ khoảng 4.8k mẫu nên thường lấy toàn bộ.
MAX_TRAIN_SAMPLES = min(
    MAX_TRAIN_SAMPLES,
    len(dataset)
)

dataset = dataset.select(
    range(MAX_TRAIN_SAMPLES)
)

dataset

Dataset({
    features: ['sentence', 'label'],
    num_rows: 4846
})

In [29]:
# Chia dữ liệu thành train / validation / test theo tỷ lệ 80% / 10% / 10%.
# stratify_by_column="label" giúp giữ phân bố nhãn tương đối đều giữa các split.
train_temp = dataset.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)

validation_test = train_temp["test"].train_test_split(
    test_size=0.5,
    seed=42,
    stratify_by_column="label"
)

dataset = DatasetDict({
    "train": train_temp["train"],
    "validation": validation_test["train"],
    "test": validation_test["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3876
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 485
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 485
    })
})


In [30]:
print(dataset["train"][0])
print(dataset["validation"][0])
print(dataset["test"][0])

{'sentence': 'Deliveries will start in the second half of 2007 and the start-up of the mill is scheduled for 2008 .', 'label': 1}
{'sentence': 'A replay will be available until 27 October 2006 in the following numbers : US callers : +1 617-á801-á6888 , non-US callers : +44 20 7365 8427 , access code : 2659 5401 .', 'label': 1}
{'sentence': 'ALEXANDRIA , Va. , Aug. 27 -- Timo Vataja of Tampere , Finland , Virve Inget of Oulu , Finland , have developed a computer program product with activating the right of use .', 'label': 1}


In [31]:
# Kiểm tra phân bố nhãn sau khi split.
from collections import Counter

for split in dataset:
    counts = Counter(dataset[split]["label"])
    print(split, {id2label[k]: v for k, v in sorted(counts.items())})

train {'negative': 483, 'neutral': 2303, 'positive': 1090}
validation {'negative': 61, 'neutral': 288, 'positive': 136}
test {'negative': 60, 'neutral': 288, 'positive': 137}


In [32]:
label2id

{'negative': 0, 'neutral': 1, 'positive': 2}

In [33]:
id2label

{0: 'negative', 1: 'neutral', 2: 'positive'}

In [ ]:
# Dataset đã sẵn sàng với các cột: sentence, label
# label là ID số nguyên theo mapping: negative=0, neutral=1, positive=2.

In [39]:
USER_PROMPT_TEMPLATE = """Predict the sentiment of the following input sentence.
The response must begin with "Sentiment: ", followed by one of these keywords: "positive", "negative", or "neutral", to reflect the sentiment of the input sentence.

Sentence: {input}"""

def tokenize_function(examples): # Định nghĩa hàm để mã hóa (tokenize) các ví dụ trong tập dữ liệu.
    results = { # Khởi tạo một từ điển để lưu trữ kết quả đầu ra.
        "input_ids": [], # Danh sách lưu trữ các ID đầu vào đã mã hóa.
        "labels": [], # Danh sách lưu trữ các ID nhãn.
        "attention_mask": [], # Danh sách lưu trữ mặt nạ chú ý.
    }

    for i in range(len(examples['sentence'])): # Lặp qua từng câu trong tập dữ liệu.
        cur_input = examples['sentence'][i] # Lấy câu đầu vào hiện tại.
        cur_output_id = examples['label'][i] # Lấy ID cảm xúc đầu ra hiện tại.

        cur_prompt = USER_PROMPT_TEMPLATE.format(input=cur_input) # Định dạng prompt người dùng với câu đầu vào hiện tại.
        cur_output = id2label[int(cur_output_id)] # Chuyển ID sang text label.

        input_messages = [ # Chuẩn bị danh sách tin nhắn cho prompt đầu vào (hệ thống và người dùng).
            {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
            {"role": "user", "content": cur_prompt},
        ]
        input_output_messages = input_messages + [{"role": "assistant", "content": f"Sentiment: {cur_output}"}] # Chuẩn bị danh sách tin nhắn bao gồm cả phản hồi của trợ lý.

        input_prompt = tokenizer.apply_chat_template(
            conversation=input_messages,
            add_generation_prompt=True,
            tokenize=False
        )

        input_output_prompt = tokenizer.apply_chat_template(
            conversation=input_output_messages,
            add_generation_prompt=False,
            tokenize=False
        )

        input_prompt_tokenized = tokenizer(
            input_prompt,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"][0]

        input_output_prompt_tokenized = tokenizer(
            input_output_prompt,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"][0]

        input_ids = input_output_prompt_tokenized # Gán ID đầu vào là tensor đã mã hóa toàn bộ cuộc trò chuyện.
        label_ids = torch.cat([ # Tạo ID nhãn, đánh dấu các phần của prompt không cần huấn luyện là -100.
            torch.full_like(input_prompt_tokenized, fill_value=-100),
            input_output_prompt_tokenized[len(input_prompt_tokenized):]
        ])

        assert len(input_ids) == len(label_ids) # Đảm bảo chiều dài của input_ids và label_ids là như nhau.

        results["input_ids"].append(input_ids) # Thêm input_ids vào kết quả.
        results["labels"].append(label_ids) # Thêm label_ids vào kết quả.
        results['attention_mask'].append(torch.ones_like(input_ids)) # Thêm mặt nạ chú ý vào kết quả.

    return results # Trả về từ điển chứa các input_ids, labels, và attention_mask đã xử lý.


col_names = dataset['train'].column_names # Lấy tên các cột từ tập dữ liệu huấn luyện.
tokenized_dataset = dataset.map( # Áp dụng hàm tokenize_function cho toàn bộ tập dữ liệu.
    tokenize_function,
    batched=True, # Xử lý theo lô.
    remove_columns=col_names, # Xóa các cột ban đầu không còn cần thiết.
    num_proc=1, # Dùng 1 process để tránh lỗi multiprocessing với tokenizer trong Colab.
)
tokenized_dataset # Hiển thị cấu trúc của tập dữ liệu đã mã hóa.

Map (num_proc=1):   0%|          | 0/3876 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/485 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/485 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 3876
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 485
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 485
    })
})

In [40]:
print(tokenized_dataset['train'][0])
print(tokenizer.decode(tokenized_dataset['train'][0]['input_ids'], skip_special_tokens=False))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 845, 3297, 220, 2366, 21, 271, 2675, 527, 264, 11190, 18328, 13, 1472, 2011, 21054, 279, 1217, 1715, 13, 128009, 128006, 882, 128007, 271, 54644, 279, 27065, 315, 279, 2768, 1988, 11914, 627, 791, 2077, 2011, 3240, 449, 330, 32458, 3904, 25, 3755, 8272, 555, 832, 315, 1521, 21513, 25, 330, 31587, 498, 330, 43324, 498, 477, 330, 60668, 498, 311, 8881, 279, 27065, 315, 279, 1988, 11914, 382, 85664, 25, 65752, 552, 690, 1212, 304, 279, 2132, 4376, 315, 220, 1049, 22, 323, 279, 1212, 5352, 315, 279, 2606, 374, 13847, 369, 220, 1049, 23, 662, 128009, 128006, 78191, 128007, 271, 32458, 3904, 25, 21277, 128009], 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

7. CUSTOM DATA COLLATOR

In [41]:
from transformers import DataCollatorWithPadding
from typing import Any, Dict, List

class RightPaddingDataCollator(DataCollatorWithPadding):
    """The default data collator pads only inputs, not including the labels."""

    def __init__(self, tokenizer, max_length: int = 1024):
        super().__init__(tokenizer, max_length=max_length)

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids, labels, attention_mask = [], [], []
        max_batch_len = max(len(f["input_ids"]) for f in features)

        for sample in features:
            # Convert to torch tensors
            cur_input_ids = torch.tensor(sample["input_ids"], dtype=torch.long)
            cur_labels = torch.tensor(sample["labels"], dtype=torch.long)
            cur_attention_mask = torch.ones_like(cur_input_ids)

            # Next, we pad the inputs and labels to the maximum length within the batch
            pad_token_id = self.tokenizer.pad_token_id
            padding_length = max_batch_len - len(cur_input_ids)
            cur_input_ids = torch.cat([cur_input_ids, torch.full((padding_length,), fill_value=pad_token_id, dtype=torch.long)])
            cur_labels = torch.cat([cur_labels, torch.full((padding_length,), fill_value=-100, dtype=torch.long)])
            cur_attention_mask = torch.cat([cur_attention_mask, torch.zeros((padding_length,), dtype=torch.long)])

            # Truncate the inputs and labels to the maximum length
            cur_input_ids = cur_input_ids[:max_batch_len]
            cur_labels = cur_labels[:max_batch_len]
            cur_attention_mask = cur_attention_mask[:max_batch_len]

            # Append to the return lists
            input_ids.append(cur_input_ids)
            labels.append(cur_labels)
            attention_mask.append(cur_attention_mask)

        # Return formatted batch.
        return {
            "input_ids": torch.stack(input_ids),
            "labels": torch.stack(labels),
            "attention_mask": torch.stack(attention_mask)
        }

data_collator = RightPaddingDataCollator(tokenizer) # Khởi tạo một đối tượng RightPaddingDataCollator với tokenizer đã được cấu hình.

In [42]:
accuracy_metric = evaluate.load("accuracy") # Tải chỉ số 'accuracy' để đánh giá hiệu suất mô hình.
f1_metric = evaluate.load("f1") # Tải chỉ số 'f1' (F1-score) để đánh giá cân bằng giữa độ chính xác và độ thu hồi.
precision_metric = evaluate.load("precision") # Tải chỉ số 'precision' để đánh giá tỷ lệ các kết quả dương tính đúng.
recall_metric = evaluate.load("recall") # Tải chỉ số 'recall' để đánh giá tỷ lệ các kết quả dương tính thực sự được xác định.


def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0] # Nếu logits là một tuple, chỉ lấy phần tử đầu tiên (thường là đầu ra chính của mô hình).
    return logits.argmax(dim=-1) # Trả về chỉ số của giá trị lớn nhất theo chiều cuối cùng, đây sẽ là dự đoán của mô hình.


def compute_metrics(eval_preds):
    preds, labels = eval_preds # Giải nén đầu ra dự đoán và nhãn thực tế.

    if isinstance(preds, tuple):
        preds = preds[0] # Nếu preds là một tuple, chỉ lấy phần tử đầu tiên.

    idx = 0
    for i in range(len(labels[0])):
        if labels[0][i] == -100:
            idx = i # Tìm vị trí đầu tiên không phải -100 (tức là bắt đầu của phần nhãn thực tế).
        else:
            break
    # Cắt nhãn và dự đoán để loại bỏ các token nhắc (prompt tokens).
    preds = preds[:, idx:]

    # Thay thế -100 trong dự đoán bằng ID token đệm (pad_token_id) vì không thể giải mã chúng.
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    processed_preds = []
    for pred in preds:
        end_pred_idx = np.where(pred == tokenizer.eos_token_id)[0] # Tìm vị trí của token kết thúc câu (eos_token_id).
        if len(end_pred_idx) > 0:
            end_pred_idx = end_pred_idx[0]
            processed_preds.append(pred[:end_pred_idx]) # Cắt dự đoán cho đến token kết thúc câu.
        else:
            processed_preds.append(pred) # Nếu không tìm thấy eos_token, giữ nguyên dự đoán.

    # Giải mã các dự đoán đã tạo thành văn bản.
    decoded_preds = tokenizer.batch_decode(processed_preds, skip_special_tokens=True)

    # Thay thế -100 trong nhãn bằng ID token đệm (pad_token_id) vì không thể giải mã chúng.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Giải mã các nhãn tham chiếu thành văn bản.
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Chuyển đổi các dự đoán và nhãn đã giải mã thành ID nhãn số nguyên.
    int_preds, int_labels = [], []
    for p, l in zip(decoded_preds, decoded_labels):
        l = l.split(":")[-1].strip() # Trích xuất nhãn thực tế sau dấu hai chấm và loại bỏ khoảng trắng.
        cur_label_id = label2id[l] # Chuyển đổi nhãn văn bản thành ID số nguyên.
        int_labels.append(cur_label_id) # Thêm ID nhãn thực tế vào danh sách.
        try:
            p = p.split(":")[-1].strip() # Trích xuất dự đoán sau dấu hai chấm và loại bỏ khoảng trắng.
            cur_pred_id = label2id[p] # Chuyển đổi dự đoán văn bản thành ID số nguyên.
        except Exception as e:
            cur_pred_id = (cur_label_id + 1) % len(label2id) # Nếu dự đoán không hợp lệ, gán ID nhãn khác ngẫu nhiên để tránh lỗi.
        int_preds.append(cur_pred_id) # Thêm ID dự đoán vào danh sách.

    # Tính toán các chỉ số đánh giá.
    accuracy_results = accuracy_metric.compute(predictions=int_preds, references=int_labels) # Tính toán độ chính xác.
    f1_results = f1_metric.compute(predictions=int_preds, references=int_labels, average="macro") # Tính toán F1-score với average="macro".
    precision_results = precision_metric.compute(predictions=int_preds, references=int_labels, average="macro") # Tính toán Precision với average="macro".
    recall_results = recall_metric.compute(predictions=int_preds, references=int_labels, average="macro") # Tính toán Recall với average="macro".

    return {
        **accuracy_results,
        **f1_results,
        **precision_results,
        **recall_results
    }

8. TRAIN MODEL

In [43]:
import bitsandbytes as bnb # Nhập thư viện bitsandbytes để tối ưu hóa việc sử dụng bộ nhớ cho các mô hình lớn.
from transformers import get_linear_schedule_with_warmup # Nhập hàm get_linear_schedule_with_warmup để tạo lịch trình tốc độ học.

trainable_params = filter(lambda p: p.requires_grad, peft_model.parameters()) # Lọc các tham số của mô hình có thể huấn luyện được.

paged_optimizer = bnb.optim.PagedAdamW( # Khởi tạo trình tối ưu PagedAdamW từ bitsandbytes, được thiết kế để xử lý các mô hình lớn hiệu quả hơn.
    trainable_params, # Truyền các tham số có thể huấn luyện vào trình tối ưu.
    lr=5e-5, # Đặt tốc độ học ban đầu. Giảm từ 3e-4 xuống 5e-5 để hạn chế overfitting trên dataset nhỏ.
    weight_decay=0.01, # Đặt hệ số suy giảm trọng số (regularization) để hạn chế overfitting.
)
scheduler = get_linear_schedule_with_warmup( # Khởi tạo lịch trình tốc độ học tuyến tính với giai đoạn khởi động.
    paged_optimizer, # Truyền trình tối ưu đã được tạo.
    num_warmup_steps=int(MAX_TRAIN_STEPS*0.05), # Đặt số bước khởi động (5% tổng số bước huấn luyện).
    num_training_steps=MAX_TRAIN_STEPS, # Đặt tổng số bước huấn luyện.
)

# trainer = Trainer(
trainer = SFTTrainer( # Khởi tạo SFTTrainer từ thư viện TRL để tinh chỉnh có giám sát.
    model=peft_model, # Truyền mô hình PEFT đã được cấu hình.
    args=training_args, # Truyền các đối số huấn luyện đã được định nghĩa.
    train_dataset=tokenized_dataset['train'], # Cung cấp tập dữ liệu huấn luyện đã được mã hóa.
    eval_dataset=tokenized_dataset['validation'], # Cung cấp tập dữ liệu đánh giá đã được mã hóa.
    preprocess_logits_for_metrics=preprocess_logits_for_metrics, # Truyền hàm xử lý logits trước khi tính toán các chỉ số.
    compute_metrics=compute_metrics, # Truyền hàm tính toán các chỉ số đánh giá.
    processing_class=tokenizer, # Chỉ định tokenizer được sử dụng để xử lý dữ liệu.
    data_collator=data_collator, # Truyền đối tượng data collator tùy chỉnh.
    optimizers=(paged_optimizer, scheduler), # Cung cấp trình tối ưu và lịch trình tốc độ học.
)
# trainer.train() # Bắt đầu quá trình huấn luyện mô hình.


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [44]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
100,0.145255,0.102514,0.035052,0.036563,0.051121,0.044666
200,0.072112,0.092039,0.039175,0.039730,0.054389,0.049568
300,0.076657,0.078344,0.041237,0.041595,0.058811,0.050725
400,0.052469,0.071232,0.041237,0.041595,0.058811,0.050725
500,0.086886,0.071447,0.043299,0.043176,0.060518,0.053176
600,0.045387,0.070010,0.041237,0.041595,0.058811,0.050725
700,0.119071,0.070443,0.041237,0.041595,0.058811,0.050725
800,0.056283,0.070410,0.043299,0.043176,0.060518,0.053176


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

TrainOutput(global_step=800, training_loss=0.09775268476456404, metrics={'train_runtime': 1172.0888, 'train_samples_per_second': 5.46, 'train_steps_per_second': 0.683, 'total_flos': 5679164628910080.0, 'train_loss': 0.09775268476456404})

In [45]:
# Evaluate the model on the validation and test set
trainer.evaluate(tokenized_dataset["validation"])
trainer.evaluate(tokenized_dataset["test"])


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.056283,0.071435,800,0.043299,0.043176,0.060518,0.053176


Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.056283,0.078331,800,0.063918,0.069811,0.107456,0.081841


{'eval_loss': 0.07833133637905121,
 'eval_accuracy': 0.06391752577319587,
 'eval_f1': 0.06981054228603058,
 'eval_precision': 0.10745554644632566,
 'eval_recall': 0.08184137604758042}

In [47]:
def inference(model, tokenizer, input_sentence):
    tokenizer.pad_token_id = tokenizer.eos_token_id # Đặt pad_token_id của tokenizer bằng eos_token_id để xử lý padding.

    user_prompt = USER_PROMPT_TEMPLATE.format(input=input_sentence) # Định dạng câu nhập liệu của người dùng vào template prompt.
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
        {"role": "user", "content": user_prompt},
    ]
    input_prompt = tokenizer.apply_chat_template(conversation=messages, add_generation_prompt=True, tokenize=False) # Áp dụng chat template để tạo prompt đầu vào hoàn chỉnh.
    inputs = tokenizer(input_prompt, return_tensors="pt", add_special_tokens=False) # Mã hóa prompt thành tensor PyTorch.
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Chuyển các tensor đầu vào sang thiết bị của mô hình (GPU/CPU).

    output_ids = model.generate(**inputs, max_new_tokens=16, pad_token_id=tokenizer.eos_token_id) # Tạo ra các token đầu ra từ mô hình.
    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]] # Cắt bỏ các token prompt khỏi đầu ra.
    results = tokenizer.batch_decode(output_ids, skip_special_tokens=True) # Giải mã các token đầu ra thành văn bản.
    return results[0] # Trả về kết quả suy luận đầu tiên.

def batch_inference(model, tokenizer, input_sentences):
    tokenizer.padding_side = "left" # Đặt padding_side thành 'left' cho suy luận theo batch.
    tokenizer.pad_token_id = tokenizer.eos_token_id # Đặt pad_token_id của tokenizer bằng eos_token_id.

    user_prompts = [USER_PROMPT_TEMPLATE.format(input=input_sentence) for input_sentence in input_sentences] # Định dạng nhiều câu nhập liệu.
    messages_list = [
        [
            {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
            {"role": "user", "content": user_prompt},
        ]
        for user_prompt in user_prompts
    ]
    input_prompts = [tokenizer.apply_chat_template(conversation=messages, add_generation_prompt=True, tokenize=False) for messages in messages_list] # Áp dụng chat template cho từng prompt.

    inputs = tokenizer(input_prompts, return_tensors="pt", padding=True, add_special_tokens=False) # Mã hóa batch các prompt thành tensor và thêm padding.
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Chuyển các tensor đầu vào sang thiết bị của mô hình.

    output_ids = model.generate(**inputs, max_new_tokens=16, pad_token_id=tokenizer.eos_token_id) # Tạo ra các token đầu ra cho batch.
    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]] # Cắt bỏ các token prompt khỏi đầu ra.
    results = tokenizer.batch_decode(output_ids, skip_special_tokens=True) # Giải mã các token đầu ra thành văn bản.
    return results # Trả về danh sách các kết quả suy luận.

In [48]:
print(inference(
    peft_model,
    tokenizer,
    "Operating profit increased by 25 percent."
))

print(inference(
    peft_model,
    tokenizer,
    "The company reported significant losses."
))

print(inference(
    peft_model,
    tokenizer,
    "The company announced a new board meeting."
))

[transformers] Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: positive


[transformers] Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: negative
Sentiment: neutral


9. PUSH MODEL TO HUGGING FACE

In [56]:
from huggingface_hub import login

login() # Dán Hugging Face token có quyền Write khi Colab yêu cầu. Không nên hardcode token trực tiếp trong notebook.

In [57]:
# Push to the hub
hub_model_name = "sanekojp1508/qlora-financial"
peft_model.push_to_hub(hub_model_name) # Đẩy mô hình PEFT đã huấn luyện lên Hugging Face Hub với tên được chỉ định.
tokenizer.push_to_hub(hub_model_name) # Đẩy tokenizer đã cấu hình lên Hugging Face Hub.

# or save to local directory
# peft_model.save_pretrained("./peft-lora-causal-lm-1b") # Lưu mô hình PEFT vào thư mục cục bộ.
# tokenizer.save_pretrained("./peft-lora-causal-lm-1b") # Lưu tokenizer vào thư mục cục bộ.

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  36%|###5      | 1.23MB / 3.42MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpklg0_mj4/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/sanekojp1508/qlora-financial/commit/54e3eb392b8ba31049d4c83cfda6d25f7c6cc50e', commit_message='Upload tokenizer', commit_description='', oid='54e3eb392b8ba31049d4c83cfda6d25f7c6cc50e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sanekojp1508/qlora-financial', endpoint='https://huggingface.co', repo_type='model', repo_id='sanekojp1508/qlora-financial'), pr_revision=None, pr_num=None)

In [58]:
merged_model = peft_model.merge_and_unload()

In [59]:
merged_model.push_to_hub(
    "sanekojp1508/qlora-financial-merged"
)

tokenizer.push_to_hub(
    "sanekojp1508/qlora-financial-merged"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...svkvq2z/model.safetensors:   0%|          | 1.11MB / 1.55GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpplmr7nux/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/sanekojp1508/qlora-financial-merged/commit/3a5d4c0cd56e622eab8e546518dae0c6266e2162', commit_message='Upload tokenizer', commit_description='', oid='3a5d4c0cd56e622eab8e546518dae0c6266e2162', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sanekojp1508/qlora-financial-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='sanekojp1508/qlora-financial-merged'), pr_revision=None, pr_num=None)